## This is the Real Time Prediction Section
Here all the models are trained and predicted real time based on camera

#### Import All files

In [3]:
import cv2
from HandTrackingModule import HandDetector
import numpy as np
import pickle

In [4]:
## Calling the Hand Detector
Detector= HandDetector()

In [5]:
# Load the Model From Model folder
with open("model/SVM_Model.pkl", 'rb') as file:  
    load_model = pickle.load(file)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


/media/shahan/New Volume/CSE_499_SIGN_LANGUAGE_RECOGNITION_2.0/CSE_499_SIGN_LANGUAGE_RECOGNITION_2.0/shahan_venv/lib/python3.11/site-packages/sklearn/base.py:347: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.2.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### Openining Window & Main Operation

In [6]:
# Default some value
offset = 20
Size = 300
count=1
loop=0
feature11=0
feature12=0
save=0
offset=20

In [9]:
cap = cv2.VideoCapture(0)

# Create a named window
cv2.namedWindow("Webcam", cv2.WINDOW_NORMAL)  # Use WINDOW_NORMAL for resizable window

# Set the initial window size
cv2.resizeWindow("Webcam", 800, 600)  # Adjust the size as needed

while True:
    ret, frame = cap.read()
    if not ret:
        break
    else:
        hands, img= Detector.findHands(frame)
        # print(hands)
        if hands:
            hand1=hands[0]
            if hand1["type"]=="Right":
                x,y,w,h=hand1["bbox"]
                # print("Right Image")
                lmList=hand1["lmList"]
                lengthBase, info1=Detector.findDistance(lmList[0][:2],lmList[12][:2])
                # print(lengthBase)
                lengthThumb, infoThumb=Detector.findDistance(lmList[0][:2],lmList[4][:2])
                # print(lengthIndex)
                ratio_BT=round(lengthThumb/lengthBase,3)

                lengthIndex, infoIndex=Detector.findDistance(lmList[0][:2],lmList[8][:2])
                # print(lengthIndex)
                ratio_BI=round(lengthIndex/lengthBase,3)

                lengthRing, infoRing=Detector.findDistance(lmList[0][:2],lmList[16][:2])
                # print(lengthIndex)
                ratio_BR=round(lengthRing/lengthBase,3)

                lengthLittle, infoLittle=Detector.findDistance(lmList[0][:2],lmList[20][:2])
                # print(lengthIndex)
                ratio_BL=round(lengthLittle/lengthBase,3)

                lengthLT, infoLT=Detector.findDistance(lmList[4][:2],lmList[20][:2])
                # print(lengthIndex)
                try:
                    ratio_BLT=round(lengthLT/lengthBase,3)
                except:
                    ratio_BLT=0
                length4to20, infol4to20=Detector.findDistance(lmList[4][:2],lmList[20][:2])
                # print(lengthIndex)
                # ratio_4_20=length4to20/lengthBase

                length8to20, info8to20=Detector.findDistance(lmList[8][:2],lmList[20][:2])
                # print(lengthIndex)
                ratio_8_20=round(length8to20/length4to20,3)

                length12to20, info12to20=Detector.findDistance(lmList[12][:2],lmList[20][:2])
                # print(lengthIndex)
                ratio_12_20=round(length12to20/length4to20,3)

                length16to20, info16to20, =Detector.findDistance(lmList[16][:2],lmList[20][:2])
                # print(lengthIndex)
                ratio_16_20=round(length16to20/length4to20,3)

                length16to12, info16to12=Detector.findDistance(lmList[16][:2],lmList[12][:2])
                # print(lengthIndex)
                ratio_16_12=round(length16to12/length4to20,3)
                
                length8to12, info8to12,=Detector.findDistance(lmList[8][:2],lmList[12][:2])
                # print(lengthIndex)
                ratio_8_12=round(length8to12/length4to20,3)
                if Detector.fingersUp(hand1)==[0,1,0,0,0] or Detector.fingersUp(hand1)==[0,1,1,0,0] or Detector.fingersUp(hand1)==[0,1,0,0,1] or Detector.fingersUp(hand1)==[0,1,1,0,1]:
                    x9 = lmList[8][0]
                    y9 = lmList[8][1]
                    angle1= np.arctan(y9/x9)
                    if angle1 >=0.90:
                        feature11=1
                        # print("Feature 1-10",feture11,feture12)
                    else:
                        feature11=0
                        # print("Feature 1-10",feture11,feture12)
                elif Detector.fingersUp(hand1)==[0,0,0,0,1]:
                    x10 = lmList[20][0]
                    y10 = lmList[20][1]
                    angle2= np.arctan(y10/x10)
                    if angle2 >=0.90:
                        feature12=1
                        # print("Feature 1-10",feature11,feature11)
                    else:
                        feature12=0
                        # print("Feature 1-10",feature11,feature11)
                else:
                    feature11=0
                    feature12=0
                    # print("Feature 1-10")

                    # print("Feature 12")

                # ratio=[ratio_BT,ratio_BI,ratio_BR,ratio_BL]
                ratio=str(ratio_BT)+','+ str(ratio_BI)+','+str(ratio_BR)+','+str(ratio_BL)+','+str(ratio_BLT)
                ratio1=str(ratio_8_20)+','+str(ratio_12_20)+','+str(ratio_16_20)+','+str(ratio_16_12)+','+str(ratio_8_12)
                ratio3= str(feature11)+','+str(feature12)

                
                total=(ratio+","+ratio1+","+ratio3)
                totalArray=np.fromstring(total, dtype=float, sep=",")
                '''
                np.fromstring('1, 2', dtype=int, sep=',')
                array([1, 2])
                '''


                prediction=load_model.predict([totalArray])
                print(prediction, end=", ")
                # score= load_model.decision_function([totalArray])
                score= load_model.predict_proba([totalArray])
                # print(score)
                maxValue=np.max(score)
                print("Max value= ",maxValue,end=",")
                max=np.argmax(score)
                print(max)
                cv2.rectangle(frame, (x - offset, y - offset-50),
                (x - offset+90, y - offset-50+50), (255, 0, 255), cv2.FILLED)
                cv2.putText(frame, str(prediction[0]), (x, y -26), cv2.FONT_HERSHEY_COMPLEX, 1.7, (255, 255, 255), 2)
                cv2.rectangle(frame, (x-offset, y-offset),
                (x + w+offset, y + h+offset), (255, 0, 255), 4)

    # Display the frame
    cv2.imshow("Webcam", frame)
    # Exit when 'q' is pressed
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release the capture and close the window
cap.release()
cv2.destroyAllWindows()

['B'], Max value=  0.40410968728885777,1
['C'], Max value=  0.36414494332010244,2
['C'], Max value=  0.3562857930167924,2
['B'], Max value=  0.3622483934798556,2
['C'], Max value=  0.37679054896400893,2
['C'], Max value=  0.5272261057112322,2
['C'], Max value=  0.6891227491796026,2
['C'], Max value=  0.5007145419670515,2
['C'], Max value=  0.5366158694305939,2
['C'], Max value=  0.5350961647844032,2
['C'], Max value=  0.43401250406674374,2
['C'], Max value=  0.4600010193383217,2
['C'], Max value=  0.5084630823029143,2
['C'], Max value=  0.5618511953715,2
['C'], Max value=  0.44974744230671915,2
['C'], Max value=  0.42368626174354074,2
['C'], Max value=  0.506342258186223,2
['A'], Max value=  0.6668137734351932,0
['A'], Max value=  0.5224910232397723,0
['A'], Max value=  0.5541402308543121,0
['A'], Max value=  0.5617676964840564,0
['A'], Max value=  0.5478758732299879,0
['A'], Max value=  0.5374949060507994,0
['A'], Max value=  0.49222674474164013,0
['A'], Max value=  0.5192310845987812